# Module 2a — Why Ingestion Matters & The Canonical Document

> **Chapter 2, Module A of 14** · [chapter index](../README.md)

> **Prerequisites:** [Chapter 1 — RAG Fundamentals](../../01_rag_fundamentals/)

> **Next:** [2b — PDF Parsing](../2b_pdf_parsing/)


## The one idea

Chapter 1 ended with `Documents → Chunk → Embed`. 

This chapter is **everything before that
first arrow** — and it is where most real-world RAG systems quietly fail.

> **A retriever cannot retrieve information that ingestion failed to capture.**

This module establishes *why* that matters and gives you the data structure the other 13
modules all depend on.

**By the end you will be able to:**

1. Explain the ingestion quality ceiling and defend it in an interview
2. Distinguish ingestion from chunking (they are constantly confused)
3. Design a canonical document schema that decouples sources from downstream code
4. Explain why `raw_text` and `clean_text` must be separate fields


## Setup

In [1]:
# `sys` lets us add the project root to Python's import path.
import sys

# The notebook sits two levels below the project root (chapter/module/notebook).
sys.path.append("../..")

# `Path` builds cross-platform filesystem paths.
from pathlib import Path

# The corpus helper downloads real public documents and caches them locally.
from utils.corpus import fetch, describe, CORPUS

# Confirm the helper loaded and show how many documents are catalogued.
print(f"Corpus helper loaded - {len(CORPUS)} real documents available.")

Corpus helper loaded - 9 real documents available.



# 1. A failure that has nothing to do with your retriever

Imagine a PDF contains:

```
Section 7.2 — Leave Policy

Employees with at least five years
of continuous service are entitled
to 30 days of annual leave.
```

But your PDF parser extracts this:

```
Page 18
Company Confidential

Employees with at least five years

Page 19
Company Confidential

of continuous service are entitled

Page 20
Company Confidential

to 30 days of annual leave.
```

Chunking then produces:

```
Chunk 1: Page 18 Company Confidential Employees with at least five years
Chunk 2: Page 19 Company Confidential of continuous service are entitled
Chunk 3: Page 20 Company Confidential to 30 days of annual leave
```

Now the retriever can **never** return the condition and the answer together. A user asking
*"Who gets 30 days of leave?"* gets Chunk 3 — and is told **every** employee gets 30 days.

Your embedding model was fine. Your vector DB was fine. Your LLM was fine. **The failure
happened three stages before any of them ran.**

## 1.1 The ceiling

$$\text{RAG Quality} \le \text{Quality of Available Parsed Knowledge}$$

Extending the model from Chapter 1:

$$Q_{\text{RAG}} \approx Q_{\text{ingestion}} \times Q_{\text{retrieval}} \times Q_{\text{generation}}$$

(Conceptual, not a formal law.) If $Q_{\text{ingestion}} = 0.4$, even a perfect retriever
and a perfect generator are capped at 0.4.

In [2]:
# Define a function that models the multiplicative quality ceiling.
def rag_quality(ingestion, retrieval, generation):

    # Each stage can only pass through what the previous stage preserved.
    return ingestion * retrieval * generation


# A team that has tuned retrieval and generation hard, but ignored ingestion.
neglected = rag_quality(ingestion=0.40, retrieval=0.95, generation=0.95)

# The same team after fixing only the parser - nothing else changed.
fixed = rag_quality(ingestion=0.90, retrieval=0.95, generation=0.95)

# Show both outcomes.
print(f"great retriever + great LLM, bad parsing : {neglected:.2f}")
print(f"same retriever + same LLM, good parsing  : {fixed:.2f}")

# Quantify the improvement from touching ONLY the ingestion stage.
print(f"\nimprovement from fixing ingestion alone  : {(fixed/neglected - 1)*100:.0f}%")

great retriever + great LLM, bad parsing : 0.36
same retriever + same LLM, good parsing  : 0.81

improvement from fixing ingestion alone  : 125%


That is the argument for this entire chapter in three numbers. Teams routinely spend weeks
tuning rerankers while a broken parser caps them at 0.36.

## 1.2 What "ingestion" means precisely

**Ingestion** is the complete process of taking external data and transforming it into a
standardized representation suitable for indexing and retrieval:

```
External data → Acquire → Parse → Understand structure → Clean
              → Normalize → Add metadata → Validate → Store canonical form
```

And a distinction that trips people up constantly:

$$\text{Ingestion} \neq \text{Chunking}$$

Chunking is one *later* stage of the indexing pipeline. Keep them separate in your head and
in your code — this chapter is ingestion, Chapter 3 is chunking.

## 1.3 What can enter a RAG system

```
                        DATA SOURCES
                             │
        ┌────────────────────┼────────────────────┐
        ▼                    ▼                    ▼
    Documents            Databases             Services
        │                    │                    │
    PDF / DOCX           SQL / NoSQL             APIs
    PPT / TXT            Warehouses              SaaS
    HTML                 Graph DB                Web
    Markdown
```

Typical enterprise reality: PDFs, DOCX, PowerPoint, Excel, CSV, JSON, XML, HTML, Markdown,
plain text, relational databases, NoSQL, SharePoint, Confluence, Google Drive, OneDrive,
S3, websites, APIs, support tickets, Slack/Teams, email, source-code repos, scanned forms,
images, tables, knowledge graphs.


# 2. The real documents we will use

Rather than inventing toy files, this chapter works on **real public documents**, each
chosen because it demonstrates a specific failure mode.

Run the cell below to see the catalogue. Nothing downloads yet — each module fetches only
what it needs, and caches it.

In [3]:
# Print the full catalogue with licences and - most importantly - what each teaches.
describe()

Real document corpus  (/Users/ajaynikumbh/Documents/00. Office/03. Projects/05. M-projects/RAG/assets/real_corpus)

[cached] bert_paper
          BERT: Pre-training of Deep Bidirectional Transformers
          source:  arXiv 1810.04805 (Devlin et al., 2019)
          licence: arXiv non-exclusive licence
          teaches: A genuine TWO-column layout: default block extraction interleaves the columns and breaks sentences mid-flow. The real reading-order problem, reproducible on a real paper.

[cached] demo_docx
          Calibre DOCX feature demonstration
          source:  calibre-ebook.com
          licence: GPL-3.0 (calibre project demo asset)
          teaches: Real Heading 1/Heading 2 hierarchy, 5 real tables, a table of contents, footnotes and list paragraphs.

[cached] gdpr_html
          Regulation (EU) 2016/679 (GDPR) - HTML edition
          source:  EUR-Lex, CELEX 32016R0679
          licence: © European Union, reuse permitted (Decision 2011/833/EU)
          teaches: The same

Note the `teaches` line on each entry. That is the selection criterion: the GDPR PDF is here
because it has **88 pages of genuine repeated headers**, not because it is a convenient PDF.

### A note on licences

Every document is public and reusable: US Government works (public domain), EU legislation
(Decision 2011/833/EU permits reuse), arXiv papers under their distribution licence, and
Apache-2.0 / GPL test assets. The corpus is **git-ignored** — you download it, you do not
redistribute it.

---
# 3. The canonical document pattern

## 3.1 The problem

Different loaders produce different shapes: PDF gives pages, DOCX gives styled paragraphs,
HTML gives a DOM, SQL gives rows, an API gives JSON.

**Do not let every downstream component understand all those formats.** That is an
N-formats × M-consumers explosion — and every new source multiplies your maintenance.

## 3.2 The fix

Convert everything into **one standard object** immediately after parsing:

```
PDF ──────┐
DOCX ─────┤
HTML ─────┤
SQL ──────┼──►  Canonical Document Schema  ──►  Same downstream pipeline
API ──────┤
Email ────┤
Wiki ─────┘
```

Everything downstream — chunker, embedder, indexer, retriever — expects exactly:

```
Document
  ├── document_id
  ├── text
  └── metadata
```

This one decision removes an enormous amount of complexity.

In [ ]:
# `dataclass` generates __init__, __repr__ and __eq__ from field declarations.
from dataclasses import dataclass, field

# Typing helpers make the schema self-documenting.
from typing import Dict, Any, Optional


# Define the canonical document representation used by the whole pipeline.
@dataclass
class Document:

    # A stable, source-derived identifier (never a fresh random UUID - see module 2i).
    document_id: str

    # The cleaned, normalized text that will eventually be chunked and embedded.
    text: str

    # Everything ABOUT the content: source, page, dates, department, permissions.
    metadata: Dict[str, Any] = field(default_factory=dict)

    # The original extraction, kept so we can re-run cleaning without re-parsing.
    raw_text: Optional[str] = None

    # A content hash, used for deduplication and change detection (module 2h).
    checksum: Optional[str] = None


# Create one normalized document object to see the shape in practice.
document = Document(
    document_id="eurlex:32016R0679",
    text="This Regulation lays down rules relating to the protection of natural persons...",
    metadata={
        "source": "eur-lex",
        "file_name": "gdpr_regulation_2016_679.pdf",
        "title": "Regulation (EU) 2016/679",
        "jurisdiction": "EU",
        "publication_date": "2016-05-04",
        "effective_date": "2018-05-25",
        "page": 1,
        "access_group": "public",
    },
)

# Show the field values one per line so the structure is easy to read.
print(f"document_id : {document.document_id}")
print(f"text        : {document.text[:60]}...")
print(f"checksum    : {document.checksum}")
print("metadata    :")
for key, value in document.metadata.items():
    print(f"    {key:18s} = {value}")

## 3.3 Why `raw_text` and `clean_text` are separate fields

This is an excellent production pattern worth adopting from day one:

```json
{
  "raw_text":   "...original extraction...",
  "clean_text": "...normalized extraction..."
}
```

**Why?** Six months from now you will discover your cleaning rule was too aggressive. With
`raw_text` preserved you can simply:

```
raw_text  →  re-run cleaning v2
```

without re-fetching or re-parsing a single source file. Without it, you re-download
everything.

A related benefit: retrieval can use `clean_text` while **citations display the original
text** exactly as the user would see it in the source document.

Let's prove the value with a real over-cleaning bug — one you will meet again in
[module 2g](../2g_cleaning_and_normalization/).

In [ ]:
# Import the regular expression module for the cleaning rule.
import re


# A plausible-looking cleaning rule: strip punctuation to "normalize" the text.
def overly_aggressive_clean(text):

    # Replace anything that is not a word character or space with a space.
    return re.sub(r"[^\w\s]", " ", text)


# A real sentence from the GDPR, containing a legal cross-reference.
legal_text = "Processing shall be lawful under Article 6(1)(a) of Regulation (EU) 2016/679."

# Store BOTH forms, exactly as the canonical schema prescribes.
legal_document = Document(
    document_id="eurlex:32016R0679#art6",
    raw_text=legal_text,
    text=overly_aggressive_clean(legal_text),
    metadata={"jurisdiction": "EU"},
)

# Show what the cleaner did to a precise legal citation.
print("raw_text  :", legal_document.raw_text)
print("clean_text:", legal_document.text)

# The citation "Article 6(1)(a)" has become "Article 6 1 a " - a different reference.
print("\ncitation survived cleaning?",
      "Article 6(1)(a)" in legal_document.text)

`Article 6(1)(a)` became `Article 6 1 a` — a legally different reference. Because we kept
`raw_text`, this is a **recoverable** mistake: change the rule, re-run cleaning, done. Had we
overwritten the original, the precise citation would be gone from our corpus permanently.

> **This is the single highest-leverage habit in this chapter.** Storage is cheap. Re-parsing
> ten million PDFs because you discovered a cleaning bug is not.

---
# 4. Source connectors

Before parsing, you have to actually *acquire* the data. A **connector** answers one
question:

> *Where is the document stored and how do I fetch it?*

```
                    Source Layer

SharePoint    S3    Database    Website    API
     │        │         │          │        │
     └────────┴─────────┴──────────┴────────┘
                        │
                        ▼
                 Connector Layer
                        │
                        ▼
                   Raw Content
```

## 4.1 Connector responsibilities

A good connector handles: **authentication, pagination, file discovery, download,
rate limits, retries, incremental updates, deletion detection, metadata acquisition,
permissions.**

**Do not put any of this inside the parser.** Separation of responsibilities:

```
Connector  →  gets bytes / records
Parser     →  understands content
```

That boundary keeps both testable. `utils/corpus.py` in this repo is itself a minimal
connector: it handles fetching, caching, user-agent headers, timeouts and error reporting —
but it knows nothing about PDF or DOCX internals.

## 4.2 The source-of-truth problem

Suppose the same policy exists in SharePoint, Google Drive, Confluence and an email
attachment. **Which is authoritative?**

Without a source strategy, RAG retrieves conflicting versions:

```
Version A: Refund = 14 days
Version B: Refund = 30 days
```

The retriever returns both. The LLM has no way to know which to trust — and you get a
hallucination or an unhelpful hedge.

Production systems must store: **source priority · effective date · version ·
publication date · status.** We handle this in
[module 2h](../2h_deduplication_and_versioning/).

### Effective date ≠ publication date

Worth flagging now because it shapes your schema. The GDPR was **published 4 May 2016** but
**applied from 25 May 2018**. Ask *"what rules applied in January 2017?"* and publication
date gives the wrong answer.

Notice the canonical document above stores both.

---
# 5. Module summary

## Key points

1. **Ingestion caps everything downstream.** $Q_{\text{RAG}} \approx Q_{\text{ing}} \times Q_{\text{ret}} \times Q_{\text{gen}}$
2. **Ingestion ≠ chunking.** Chunking is a later stage (Chapter 3).
3. **One canonical schema** decouples N sources from M consumers.
4. **Keep `raw_text` alongside `clean_text`** — cleaning bugs become recoverable.
5. **Connector fetches bytes; parser understands content.** Keep the boundary clean.
6. **Store effective date and publication date separately.**

## Interview answer

> Ingestion is the stage that converts heterogeneous external sources into a standardized,
> metadata-rich canonical representation suitable for indexing. It matters because retrieval
> quality is bounded by it — a retriever cannot retrieve what parsing failed to capture. I
> normalize every source into one schema with a stable document ID, the cleaned text, the
> original raw text, and metadata carrying provenance, temporal fields and permissions.

## Exercises

1. **Quantify your own ceiling.** If your parser drops 15% of table content and tables hold
   30% of your answers, what is your maximum achievable Recall@K?
2. **Extend the schema.** Add the fields you would need to support citations of the form
   *"GDPR, Article 17(1)(a), page 43"*. Which module of this chapter populates each?
3. **Find a second over-cleaning victim.** Write a rule that damages `ISO/IEC 27001:2022`
   or `C++`, then show `raw_text` recovers it.

---

## → Next: [Module 2b — PDF Parsing](../2b_pdf_parsing/)

We take the three real PDFs from the corpus — an academic two-column paper, a government
form, and 88 pages of EU legislation — and watch each one break a different naive assumption.